# CUAD Exploration Notebook

Interactive viewer for the CUAD dataset and saved run logs (`output/*.md`). This notebook
reuses the extraction/scoring functions from `phase2.2_governing_law.py` directly instead
of duplicating them, so it's always consistent with the current pipeline.

**This notebook is for browsing data, not a replacement for the versioned
`phase1.x`/`phase2.x` scripts** -- those stay as version-controlled, CLI-runnable `.py`
files (clean diffs, `--n`/`--indices`/`--absent` flags, reproducible runs). Use this
notebook to look at contracts, snippets, and past results more easily than scrolling
through a terminal dump or a 400-line markdown file.

In [3]:
import json
import re
import importlib.util
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 100)


def load_module(path, name):
    """Load one of the phaseX.Y_governing_law.py scripts as a module.

    They have dots in their filenames (phase2.2_governing_law.py), so a plain
    `import` doesn't work -- this loads by file path instead.
    """
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


# Reuse the current pipeline's functions instead of reimplementing them here.
p22 = load_module("phase2.2_governing_law.py", "p22")

## Configuration

Set this to your local copy of `CUAD_v1.json` (not bundled in the repo -- see the README).

In [4]:
CUAD_JSON_PATH = "/Users/raymondcromwell/Downloads/data/CUADv1.json"

## Load contracts into DataFrames (present vs. absent Governing Law clause)

In [5]:
present = p22.load_governing_law_examples(CUAD_JSON_PATH, require_present=True)
absent = p22.load_governing_law_examples(CUAD_JSON_PATH, require_present=False)


def to_df(examples):
    return pd.DataFrame(
        [(pos, title, len(context), gold) for pos, title, context, gold in examples],
        columns=["position", "title", "context_chars", "gold"],
    )


present_df = to_df(present)
absent_df = to_df(absent)
print(f"{len(present_df)} contracts with a Governing Law clause, {len(absent_df)} without")
present_df.head(10)

437 contracts with a Governing Law clause, 73 without


,position,title,context_chars,gold
0,1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT,54290,This Agreement is to be construed according to the laws of the State of Illinois.
1,2,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION AND DISTRIBUTION AGREEMENT",70383,This Agreement is governed by English law and the parties submit to the exclusive jurisdiction o...
2,3,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_EX-10.16_Supply Agreement,11475,"It will be governed by the law of the People's Republic of China ,otherwise it is governed by Un..."
3,4,CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WEB SITE HOSTING AGREEMENT,15176,"This Agreement was entered into in the State of Florida, and its validity, construction, interpr..."
4,5,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT AGREEMENT,24632,This Agreement shall be governed and construed according to the laws of the State of Kansas.
5,6,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSULTING AGREEMENT",18403,"This Agreement shall be governed by the laws of the State of Texas, without reference to its con..."
6,7,"VEONEER,INC_02_21_2020-EX-10.11-JOINT VENTURE AGREEMENT",8257,This Amendment shall be governed by and construed in accordance with the laws of Japan.
7,8,DovaPharmaceuticalsInc_20181108_10-Q_EX-10.2_11414857_EX-10.2_Promotion Agreement,175580,This Agreement and any and all matters arising directly or indirectly herefrom shall be governed...
8,9,"PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC LICENSING, DISTRIBUTION AND MARKETING AGREEMENT",145168,"This Agreement and the relationship between the Parties shall be governed by, and interpreted in..."
9,10,"MetLife, Inc. - Remarketing Agreement",111249,THIS REMARKETING AGREEMENT AND THE PRICING AGREEMENT SHALL BE GOVERNED BY AND CONSTRUED IN ACCOR...


## Browse and filter

A couple of examples -- this is the kind of thing that's painful to do by eye in a terminal
dump but trivial with a DataFrame.

In [6]:
# Longest contracts first -- these are where retrieval/windowing matters most
present_df.sort_values("context_chars", ascending=False).head(10)

,position,title,context_chars,gold
387,388,MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSOURCING AGREEMENT,338211,This Agreement and the rights and obligations of the parties hereto shall be construed in ...
162,163,"CERES,INC_01_25_2012-EX-10.20-Collaboration Agreement",335282,"This Agreement shall be governed by, and construed and interpreted in accordance with, the laws ..."
407,408,"GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Franchise Agreement",300768,The Franchise Agreement requires application of the laws of the State of Texas.
154,155,PhasebioPharmaceuticalsInc_20200330_10-K_EX-10.21_12086810_EX-10.21_Development Agreement,291873,"The construction and validity of this Agreement and the provisions hereof, and the rights and ob..."
48,49,VerizonAbsLlc_20200123_8-K_EX-10.4_11952335_EX-10.4_Service Agreement,289615,"THIS AGREEMENT, INCLUDING THE RIGHTS AND DUTIES OF THE PARTIES HERETO, SHALL BE GOVERNED BY, AND..."
220,221,"Array BioPharma Inc. - LICENSE, DEVELOPMENT AND COMMERCIALIZATION AGREEMENT",272018,"This Agreement and all questions regarding its validity or interpretation, or the breach or perf..."
224,225,RevolutionMedicinesInc_20200117_S-1_EX-10.1_11948417_EX-10.1_Development Agreement,266213,This Agreement shall be governed by and construed in accordance with the laws of the State of Ne...
110,111,HarpoonTherapeuticsInc_20200312_10-K_EX-10.18_12051356_EX-10.18_Development Agreement,249551,"This Agreement or the performance, enforcement, breach or termination hereof shall be interprete..."
139,140,MRSFIELDSORIGINALCOOKIESINC_01_29_1998-EX-10-FRANCHISE AGREEMENT,249359,"Except to the extent governed by the U.S. Trademark Act of 1946 (Lanham Act, 15 U.S.C. ""1051 et..."
76,77,AzulSa_20170303_F-1A_EX-10.3_9943903_EX-10.3_Maintenance Agreement1,233248,relating to the subject matter of this Agreement or the transactions contemplated hereby or the ...


In [7]:
# Search gold text for a jurisdiction keyword
present_df[present_df["gold"].str.contains("Delaware", case=False)]

,position,title,context_chars,gold
12,13,CerenceInc_20191002_8-K_EX-10.4_11827494_EX-10.4_Intellectual Property Agreement,62170,"Any disputes relating to, arising out of or resulting from this Agreement, including to its exec..."
18,19,BONTONSTORESINC_04_20_2018-EX-99.3-AGENCY AGREEMENT,161626,This Agreement shall be governed by and interpreted in accordance with the laws of the State of ...
22,23,ZEBRATECHNOLOGIESCORP_04_16_2014-EX-10.1-INTELLECTUAL PROPERTY AGREEMENT,127205,The Laws of the State of Delaware (without reference to its principles of conflicts of law) shal...
38,39,ImpresseCorp_20000322_S-1A_EX-10.11_5199234_EX-10.11_Co-Branding Agreement,47639,This Agreement shall be governed by and interpreted under the laws of the State of Delaware with...
41,42,CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement,28804,"This Agreement will be governed in all respects by the laws of the State of Delaware, including ..."
62,63,UpjohnInc_20200121_10-12G_EX-2.6_11948692_EX-2.6_Manufacturing Agreement_ Supply Agreement,225094,"This Agreement and all Actions (whether in contract or tort) that may be based upon, arise out o..."
75,76,NANOPHASETECHNOLOGIESCORP_11_01_2005-EX-99.1-DISTRIBUTOR AGREEMENT,48612,This Agreement shall be governed by and interpreted under and in accordance with the laws of the...
83,84,ArconicRolledProductsCorp_20191217_10-12B_EX-2.7_11923804_EX-2.7_Trademark License Agreement,18069,This Agreement shall be governed by and interpreted in accordance with the laws of the State of ...
93,94,GULFSOUTHMEDICALSUPPLYINC_12_24_1997-EX-4-AFFILIATE AGREEMENT,27015,This Affiliate Agreement shall be governed by the laws of the State of Delaware.
94,95,"BELLRINGBRANDS,INC_02_07_2020-EX-10.18-MASTER SUPPLY AGREEMENT",37876,This Agreement will be governed by the laws of the State of Delaware without regard to its confl...


## View one contract's full text, with the gold clause highlighted

In [6]:
def show_contract(examples, position):
    _, title, context, gold = next(e for e in examples if e[0] == position)
    idx = context.find(gold.strip()[:40]) if gold else -1
    display(Markdown(f"### {title}"))
    if idx == -1:
        print(context)
    else:
        print(context[:idx])
        display(Markdown(f"**>>> {context[idx:idx + len(gold)]} <<<**"))
        print(context[idx + len(gold):])


show_contract(present, 1)

### LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT

EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has or  will  have  the facil

**>>> This Agreement is to be construed according to the laws          of the State of Illinois. <<<**



7.       NEW PRODUCTS

         7.1      Right of Option.  Should Company  introduce  other products or                   devices as contemplated by recital paragraph "A",  Distributor                   shall  have  the  option  of  becoming   Company's   exclusive                   distributor  of such  other  Products  or  devices  within the                   Market.

         7.2      Exercise of Option.  Distributor  shall exercise its option to                   become  exclusive  Distributor of other Products or devices by                   serving  written  notification  on Company of its  election to                   become  exclusive  distributor  within  thirty  (30) days upon                   which  Company  informed  Distributor  in writing of Company's                   intention  to  introduce   other   Products  or  devices.   If                   Distributor  does not exercise its option as herein  provided,                   Company may  distribute  the other  Pro

## Inspect candidate snippets -- what the extraction pipeline actually sends to Claude

In [8]:
def show_snippets(examples, position):
    _, title, context, gold = next(e for e in examples if e[0] == position)
    snippets, used_section_match = p22.build_candidate_snippets(context)
    sent = sum(len(s) for s in snippets)
    print(f"{title}\nsection-match: {used_section_match} | {len(snippets)} snippet(s) | "
          f"{sent}/{len(context)} chars sent ({sent / len(context):.0%})")
    for i, s in enumerate(snippets, 1):
        display(Markdown(f"**Snippet {i}:**"))
        print(s[:1000])


# BorrowMoney -- the contract with three competing Florida-law clauses discussed in the project
show_snippets(present, 22)

BORROWMONEYCOM,INC_06_11_2020-EX-10.1-JOINT VENTURE AGREEMENT
section-match: True | 3 snippet(s) | 4632/21450 chars sent (22%)


**Snippet 1:**

Parkway Suite H Suwanee, GA 30024 (individually the "Member" and collectively the "Members"). BACKGROUND: A. The Members wish to enter into an association of mutual benefit and agree to jointly invest and set up a joint venture enterprise. B. This Agreement sets out the terms and conditions governing this association. IN CONSIDERATION OF and as a condition of the Members entering into this Agreement and other valuable consideration, the receipt and sufficiency of which consideration is acknowledged, the Members agree as follows: Formation 1. By this Agreement the Members enter into a joint venture (the "Venture") in accordance with the laws of the State of Florida. The rights and obligations of the Members will be as stated in the applicable legislation of the State of Florida (the "Act") except as otherwise provided here. Name 2. The business name of the Venture will be BM&V2GO. Page 1 of 13





Purpose 3. The exclusive purpose of the Venture (the "Purpose") will be IT Development. i

**Snippet 2:**

of Good Faith 52. Members will use their best efforts, fairly and in good faith to facilitate the success of the Venture. Joint Venture Property 53. Where allowed by statute, title to all Venture property, including intellectual property, will remain in the name of the Venture. Where joint ventures are not recognized by statute as separate legal entities, Venture property, including intellectual property, will be held in the name of one or more Members. In all cases Venture property will be applied by the Members exclusively for the benefit and purposes of the Venture and in accordance with this Agreement. Jurisdiction 54. The Members submit to the jurisdiction of the courts of the State of Florida for the enforcement of this Agreement and for any arbitration award or decision arising from this Agreement. Page 10 of 13





Mediation and Arbitration 55. In the event a dispute arises out of, or in connection with, this Agreement, the Members will attempt to resolve the dispute through f

**Snippet 3:**

this Agreement. 62. This Agreement may be executed in counterparts. Facsimile signatures are binding and are considered to be original signatures. 63. Headings are inserted for the convenience of the Members only and are not to be considered when interpreting this Agreement. Words in the singular mean and include the plural and vice versa. Words in the masculine gender include the feminine gender and vice versa. Words in the neuter gender include the masculine gender and the feminine gender and vice versa. 64. If any term, covenant, condition or provision of this Agreement is held by a court of competent jurisdiction to be invalid, void or unenforceable, it is the Members' intent that such provision be reduced in scope by the court only to the extent deemed necessary by that court to render the provision reasonable and enforceable and the remainder of the provisions of this Agreement will in no way be affected, impaired or invalidated as a result. 65. This Agreement contains the entire

## Parse a saved run log (`output/*.md`) into a DataFrame

Works on both the older format (`phase1.1`/`1.2`, no `(contract #N)` numbering) and the
current format (`1.3`+).

In [9]:
RUN_ENTRY_RE = re.compile(
    r"\[\s*(?P<run_i>\d+)/\d+\]\s*(?:\(contract #(?P<pos>\d+)\)\s*)?(?P<status>OK|MISS)\s*(?P<title>.+?)\n"
    r"\s*gold\s*:\s*(?P<gold>.+)\n"
    r"\s*pred\s*:\s*(?P<pred>.+)\n"
    r"(?:\s*pred matched patterns:.*\n)?"
    r"\s*src\s*:\s*(?P<src>.+)\n"
    r"\s*ctx\s*:\s*(?P<ctx_full>[\d,]+) chars -> (?P<ctx_sent>[\d,]+) chars sent\n"
    r"(?:\s*usage\s*:\s*in=(?P<usage_in>\d+) out=(?P<usage_out>\d+) total=(?P<usage_total>\d+))?"
)


def parse_run_log(path):
    text = Path(path).read_text()
    rows = [m.groupdict() for m in RUN_ENTRY_RE.finditer(text)]
    df = pd.DataFrame(rows)
    for col in ["run_i", "pos", "ctx_full", "ctx_sent", "usage_in", "usage_out", "usage_total"]:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")
    return df


log_df = parse_run_log("output/phase1.3_output_50_samples.md")
print(f"parsed {len(log_df)} rows, {(log_df['status'] == 'MISS').sum()} misses")
log_df.head()

parsed 50 rows, 0 misses


,run_i,pos,status,title,gold,pred,src,ctx_full,ctx_sent,usage_in,usage_out,usage_total
0,1,1,OK,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGR...,'This Agreement is to be construed according to the laws of the State of','This Agreement is to be construed according to the laws of the State of Illinois',section-match,54290,6243,1683,20,1703
1,2,2,OK,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION ...",'This Agreement is governed by English law and the parties submit to the exclusiv','11.10 Governing Law. This Agreement is governed by English law and the parties s',section-match,70383,31973,7299,94,7393
2,3,3,OK,LohaCompanyltd_20191209_F-1_EX-10.16_11917878...,"""It will be governed by the law of the People's Republic of China ,otherwise it i""","""It will be governed by the law of the People's Republic of China ,otherwise it i""",section-match,11475,1321,541,36,577
3,4,4,OK,CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-W...,"'This Agreement was entered into in the State of Florida, and its validity, const'","'This Agreement was entered into in the State of Florida, and its validity, const'",section-match,15176,2586,773,53,826
4,5,5,OK,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT ...,'This Agreement shall be governed and construed according to the laws of the Stat','This Agreement shall be governed and construed according to the laws of the Stat',section-match,24632,112,266,21,287


In [11]:
# Just the misses
log_df[log_df["status"] == "MISS"]

,run_i,pos,status,title,gold,pred,src,ctx_full,ctx_sent,usage_in,usage_out,usage_total


In [12]:
# Most expensive contracts by input tokens
log_df.sort_values("usage_in", ascending=False).head(10)[
    ["pos", "title", "status", "usage_in", "ctx_full", "ctx_sent"]
]

,pos,title,status,usage_in,ctx_full,ctx_sent
8,9,"PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC ...",OK,28133,145168,119404
18,19,BONTONSTORESINC_04_20_2018-EX-99.3-AGENCY AGR...,OK,25988,161626,111089
9,10,"MetLife, Inc. - Remarketing Agreement",OK,21649,111249,98213
48,49,VerizonAbsLlc_20200123_8-K_EX-10.4_11952335_E...,OK,15173,289615,63749
13,14,ReynoldsConsumerProductsInc_20191115_S-1_EX-1...,OK,13018,61710,60691
27,28,CORIOINC_07_20_2000-EX-10.5-LICENSE AND HOSTI...,OK,11149,62324,52814
22,23,ZEBRATECHNOLOGIESCORP_04_16_2014-EX-10.1-INTE...,OK,9399,127205,40538
19,20,"ON2TECHNOLOGIES,INC_11_17_2006-EX-10.3-SUPPOR...",OK,9148,61392,39313
37,38,NICELTD_06_26_2003-EX-4.5-OUTSOURCING AGREEME...,OK,8890,148163,41924
43,44,CUROGROUPHOLDINGSCORP_05_04_2020-EX-10.3-SERV...,OK,7870,36602,34031


## Precision / recall / F1 across both populations

Present-clause accuracy and absent-clause correct-abstention are two numbers on two
different denominators -- that makes a tradeoff like Phase 2.2's (8 forum/venue confusions
fixed, 1 new miss on BorrowMoney) something you have to eyeball. Unifying both populations
into one confusion matrix gives an objective answer to whether the tradeoff was worth it.

Uses the saved run logs directly -- no new API calls needed.

In [13]:
def classify_population(gold_repr):
    """Absent-clause rows always print gold as the empty-string repr (`''`)."""
    return "absent" if gold_repr.strip() == "''" else "present"


def precision_recall_f1(log_path):
    df = parse_run_log(log_path)
    df["population"] = df["gold"].apply(classify_population)

    tp = ((df.population == "present") & (df.status == "OK")).sum()
    fn = ((df.population == "present") & (df.status == "MISS")).sum()
    tn = ((df.population == "absent") & (df.status == "OK")).sum()
    fp = ((df.population == "absent") & (df.status == "MISS")).sum()

    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else float("nan")

    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": precision, "recall": recall, "f1": f1}


metrics_21 = precision_recall_f1("output/phase2.1_output_hallucination_rate.md")
metrics_22 = precision_recall_f1("output/phase2.2_output_hallucination_rate.md")

pd.DataFrame(
    [{"version": "2.1", **metrics_21}, {"version": "2.2", **metrics_22}]
).set_index("version")

,TP,FP,FN,TN,precision,recall,f1
version,,,,,,,
2.1,50,10,0,63,0.833333,1.00,0.909091
2.2,49,2,1,71,0.960784,0.98,0.970297


## (Optional, costs real API credits) Run extraction live on a chosen contract

Uncomment to actually call the API.

In [ ]:
# from anthropic import Anthropic
# client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment / .env
#
# pos, title, context, gold = present[0]
# pred, usage, used_section_match = p22.extract_governing_law(client, context)
# print(f"title: {title}\ngold: {gold!r}\npred: {pred!r}\nusage: {usage}")